# Week 7
# Constraint-aware Keep-Ratio Diagnostics

This notebook imports cached data from the previous notebooks and redraws the Week 5 diagnostics across cached `(DCT window, bin size)` configurations.

Figures are produced under two constraint modes:

1. **Mean bin accuracy constraint**: use the Week 4 `binned_count_accuracy_macro` metric when a matching ratio table exists. This metric is computed by scoring each non-empty neuron-bin first, then averaging.
2. **All-neuron constraint**: use Week 5 per-neuron tables and require every baseline-spiking neuron to satisfy the target accuracy.

The template projection removes absolute probe depth by centering each neuron's local multi-channel template on its dominant channel. The first two local spatiotemporal-shape PCs are plotted as x and y; peak depth is shown as color.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import Markdown, display

ROOT = Path('G:/academic')
DATA_ROOT = ROOT / '.test_data'
BASE_RESULTS = DATA_ROOT / 'kilosort4'

TARGET_ACCURACY = 0.95
PREFERRED_SELECTION_RATIO = 0.30
MIN_BASELINE_SPIKES = 1
WEEK5_REFERENCE_BASELINE_RATIO = 1.00

RATIO_ORDER = [0.10, 0.20, 0.30, 0.50, 0.70, 1.00]

ops_base = np.load(BASE_RESULTS / 'ops.npy', allow_pickle=True).item()
templates_base = np.load(BASE_RESULTS / 'templates.npy')
print('ROOT:', ROOT)
print('templates:', templates_base.shape)
print('target accuracy:', TARGET_ACCURACY)

## Cache Discovery

The discovery logic intentionally keeps both modern `ratio1baseline` caches and older legacy caches. For a given `(DCT window, bin size)` configuration, modern `ratio1baseline` caches are preferred when they exist; otherwise the legacy cache family is used so older configurations are still visible.

In [ ]:
def week5_reference_suffix():
    return '_ratio1baseline'


def safe_config_tag(text):
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(text))


def discover_week5_neuron_caches(root=ROOT):
    rows = []
    for cache_path in sorted(Path(root).glob('week5_neuron_sensitivity_ratio_*.npy')):
        if cache_path.stat().st_size == 0:
            continue
        try:
            payload = np.load(cache_path, allow_pickle=True).item()
        except Exception as exc:
            print(f'skip unreadable cache {cache_path.name}: {exc}')
            continue
        if not isinstance(payload, dict) or 'per_neuron_table' not in payload:
            continue

        ratio = float(payload.get('ratio', np.nan))
        bin_size_samples = int(payload.get('bin_size_samples', -1))
        sample_rate = float(payload.get('sample_rate', np.nan))
        dct_label = payload.get('dct_window_label', 'unknown_dct')
        if not (np.isfinite(ratio) and bin_size_samples > 0 and np.isfinite(sample_rate)):
            continue

        is_ratio1baseline = cache_path.name.endswith('_ratio1baseline.npy') or payload.get('reference_baseline_ratio') == WEEK5_REFERENCE_BASELINE_RATIO
        rows.append({
            'ratio': ratio,
            'bin_size_samples': bin_size_samples,
            'bin_size_seconds': bin_size_samples / sample_rate,
            'sample_rate': sample_rate,
            'dct_window_label': dct_label,
            'pipeline_version': str(payload.get('pipeline_version', 'unknown_pipeline')),
            'baseline_mode': 'ratio1baseline' if is_ratio1baseline else 'legacy',
            'cache_path': cache_path,
            'payload': payload,
        })
    return pd.DataFrame(rows)


def select_cache_family_for_config(config_cache_df):
    preferred = config_cache_df[config_cache_df['baseline_mode'] == 'ratio1baseline'].copy()
    selected = preferred if not preferred.empty else config_cache_df.copy()
    selected = selected.copy()
    selected['dedupe_score'] = 0
    selected.loc[selected['baseline_mode'] == 'ratio1baseline', 'dedupe_score'] += 100
    selected.loc[selected['pipeline_version'].str.contains('v4', na=False), 'dedupe_score'] += 25
    selected.loc[selected['cache_path'].astype(str).str.contains('sampbin', regex=False), 'dedupe_score'] += 10
    selected = selected.sort_values(['ratio', 'dedupe_score', 'cache_path'], ascending=[True, False, True])
    return selected.drop_duplicates(subset=['ratio'], keep='first').drop(columns=['dedupe_score'])


week5_cache_index_df = discover_week5_neuron_caches(ROOT)
if week5_cache_index_df.empty:
    raise FileNotFoundError('No Week 5 neuron sensitivity caches found under ROOT.')

selected_config_rows = []
for config_key, raw_df in week5_cache_index_df.groupby(['dct_window_label', 'bin_size_samples', 'sample_rate'], sort=True):
    selected_df = select_cache_family_for_config(raw_df).sort_values('ratio')
    selected_config_rows.append({
        'dct_window_label': config_key[0],
        'bin_size_samples': int(config_key[1]),
        'bin_size_seconds': int(config_key[1]) / float(config_key[2]),
        'sample_rate': float(config_key[2]),
        'baseline_mode': '+'.join(sorted(selected_df['baseline_mode'].unique())),
        'ratios': sorted(selected_df['ratio'].astype(float).tolist()),
        'n_ratio_tables': int(len(selected_df)),
    })

config_summary_df = pd.DataFrame(selected_config_rows)
display(config_summary_df)

## Improved Maximum-Amplitude Projection

The old projection used PCA on flattened template waveforms. Because sparse extracellular templates are dominated by a few high-amplitude channels and polarity/scale effects, flattened PCA can create cross-like axis artifacts.

The projection uses one spike-template feature and one probe-depth coordinate:

- `x = centered local multi-channel template shape PC1`
- `y = centered local multi-channel template shape PC2`
- color = maximum template peak-to-peak amplitude
- size = total template energy

This keeps the plot as a two-component template projection while using peak depth to weight the projection.


In [ ]:
LOCAL_CHANNEL_RADIUS = 12


def get_probe_depth(ops, n_channels):
    probe = ops.get('probe', {}) if isinstance(ops, dict) else {}
    yc = probe.get('yc', None)
    if yc is None:
        yc = ops.get('yc', None) if isinstance(ops, dict) else None
    if yc is None:
        yc = np.arange(n_channels, dtype=np.float32)
    yc = np.asarray(yc, dtype=np.float32).reshape(-1)[:n_channels]
    if yc.size < n_channels:
        yc = np.pad(yc, (0, n_channels - yc.size), mode='edge')[:n_channels]
    return yc


def extract_centered_local_templates(templates, radius=LOCAL_CHANNEL_RADIUS):
    n_templates, n_time, n_channels = templates.shape
    width = 2 * radius + 1
    local_templates = np.zeros((n_templates, n_time, width), dtype=np.float32)
    dominant_channels = []
    peak_samples = []
    trough_samples = []
    template_ptp_max = []
    template_ptp_mean = []
    template_energy = []

    for neuron_id in range(n_templates):
        tpl = templates[neuron_id].astype(np.float32, copy=False)
        channel_ptp = np.ptp(tpl, axis=0)
        dominant_channel = int(np.argmax(channel_ptp))
        src_start = max(0, dominant_channel - radius)
        src_stop = min(n_channels, dominant_channel + radius + 1)
        dst_start = radius - (dominant_channel - src_start)
        dst_stop = dst_start + (src_stop - src_start)
        local_templates[neuron_id, :, dst_start:dst_stop] = tpl[:, src_start:src_stop]

        dominant_waveform = tpl[:, dominant_channel]
        dominant_channels.append(dominant_channel)
        peak_samples.append(int(np.argmax(np.abs(dominant_waveform))))
        trough_samples.append(int(np.argmin(dominant_waveform)))
        template_ptp_max.append(float(channel_ptp.max()))
        template_ptp_mean.append(float(channel_ptp.mean()))
        template_energy.append(float(np.mean(np.square(local_templates[neuron_id]))))

    return local_templates, dominant_channels, peak_samples, trough_samples, template_ptp_max, template_ptp_mean, template_energy


def local_template_shape_pca(local_templates):
    features = local_templates.reshape(local_templates.shape[0], -1).astype(np.float32, copy=False)
    features = features - features.mean(axis=1, keepdims=True)
    scale = np.linalg.norm(features, axis=1, keepdims=True)
    scale[scale <= 0] = 1.0
    features = features / scale

    centered = features - features.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    scores = centered @ vt[:2].T

    center_waveforms = local_templates[:, :, local_templates.shape[2] // 2]
    trough_sample = np.argmin(center_waveforms, axis=1)
    if np.corrcoef(scores[:, 0], trough_sample)[0, 1] < 0:
        scores[:, 0] *= -1
    if np.nanmedian(scores[:, 1]) < 0:
        scores[:, 1] *= -1
    return scores[:, 0].astype(np.float32), scores[:, 1].astype(np.float32)


def build_template_shape_projection_df(templates, ops, radius=LOCAL_CHANNEL_RADIUS):
    n_templates, _, n_channels = templates.shape
    depth_by_channel = get_probe_depth(ops, n_channels)
    local_templates, dominant_channels, peak_samples, trough_samples, template_ptp_max, template_ptp_mean, template_energy = extract_centered_local_templates(templates, radius=radius)
    template_shape_pc1, template_shape_pc2 = local_template_shape_pca(local_templates)
    peak_depths = [float(depth_by_channel[channel]) for channel in dominant_channels]

    return pd.DataFrame({
        'neuron_id': np.arange(n_templates, dtype=int),
        'template_shape_pc1': template_shape_pc1,
        'template_shape_pc2': template_shape_pc2,
        'peak_depth_um': peak_depths,
        'dominant_channel': dominant_channels,
        'local_channel_radius': radius,
        'dominant_peak_sample': peak_samples,
        'dominant_trough_sample': trough_samples,
        'template_ptp_max': template_ptp_max,
        'template_ptp_mean': template_ptp_mean,
        'template_energy': template_energy,
    })


template_projection_df = build_template_shape_projection_df(templates_base, ops_base)
display(template_projection_df.head())


## Metric Helpers

Two constraint modes are used below:

- `mean_bin`: overall metric is Week 4 `binned_count_accuracy_macro` when a matching ratio table/cache is available. This is the per-neuron-bin accuracy averaged over non-empty bins.
- `all_neuron`: overall metric is the minimum per-neuron accuracy among neurons with at least `MIN_BASELINE_SPIKES` baseline spikes.

In [ ]:
def week4_ratio1baseline_csv_path_for_config(dct_label, bin_size_samples, sample_rate, root=ROOT):
    if dct_label == 'full_timeline':
        if int(bin_size_samples) == int(round(sample_rate)):
            return root / 'week4_fixed_template_ratio_compare_ratio1baseline.csv'
        return root / f'week4_fixed_template_ratio_compare_{int(bin_size_samples)}samp_ratio1baseline.csv'
    if dct_label == 'windowed_600_samples':
        if int(bin_size_samples) == 600:
            return root / 'week4_fixed_template_ratio_compare_dctwin_600samp_600samp_ratio1baseline.csv'
        return root / f'week4_fixed_template_ratio_compare_dctwin_600samp_{int(bin_size_samples)}samp_ratio1baseline.csv'
    return None


def legacy_week4_ratio_csv_path_for_config(dct_label, bin_size_samples, sample_rate, root=ROOT):
    if dct_label == 'full_timeline':
        if int(bin_size_samples) == int(round(sample_rate)):
            return root / 'week4_fixed_template_ratio_compare.csv'
        return root / f'week4_fixed_template_ratio_compare_{int(bin_size_samples)}samp.csv'
    if dct_label == 'windowed_600_samples' and int(bin_size_samples) == 600:
        return root / 'week4_fixed_template_ratio_compare_dctwin_600samp_600samp.csv'
    return None


def load_ratio1baseline_week4_metrics_for_config(dct_label, bin_size_samples, sample_rate, root=ROOT):
    csv_path = week4_ratio1baseline_csv_path_for_config(dct_label, bin_size_samples, sample_rate, root=root)
    if csv_path is None or not csv_path.exists():
        return pd.DataFrame(columns=['ratio', 'constraint_mode', 'constraint_metric', 'metric_source'])

    df = pd.read_csv(csv_path)
    metric_col = None
    if 'binned_count_accuracy_macro_vs_ratio100' in df.columns:
        metric_col = 'binned_count_accuracy_macro_vs_ratio100'
    elif 'binned_count_accuracy_macro' in df.columns:
        metric_col = 'binned_count_accuracy_macro'
    if 'ratio' not in df.columns or metric_col is None:
        return pd.DataFrame(columns=['ratio', 'constraint_mode', 'constraint_metric', 'metric_source'])

    out = df[['ratio', metric_col]].copy()
    out['ratio'] = out['ratio'].astype(float)
    out['constraint_mode'] = 'mean_bin'
    out['constraint_metric'] = out[metric_col].astype(float)
    out['metric_source'] = str(csv_path)
    return out[['ratio', 'constraint_mode', 'constraint_metric', 'metric_source']]


def synthesize_ratio1baseline_mean_bin_from_week5(per_ratio_tables):
    rows = []
    for ratio, table_df in sorted(per_ratio_tables.items()):
        if table_df.empty or 'binned_count_accuracy' not in table_df.columns:
            continue
        valid = table_df[table_df['baseline_spikes'] > 0].copy()
        metric = float(valid['binned_count_accuracy'].astype(float).mean()) if not valid.empty else np.nan
        rows.append({
            'ratio': float(ratio),
            'constraint_mode': 'mean_bin',
            'constraint_metric': metric,
            'metric_source': 'week5_per_neuron_cache_mean_ratio1baseline',
        })
    return pd.DataFrame(rows, columns=['ratio', 'constraint_mode', 'constraint_metric', 'metric_source'])


def load_legacy_week4_ratio_metrics_for_config(dct_label, bin_size_samples, sample_rate, root=ROOT):
    csv_path = legacy_week4_ratio_csv_path_for_config(dct_label, bin_size_samples, sample_rate, root=root)
    if csv_path is not None and csv_path.exists():
        df = pd.read_csv(csv_path)
        if 'ratio' in df.columns and 'binned_count_accuracy_macro' in df.columns:
            df = df.copy()
            df['ratio'] = df['ratio'].astype(float)
            df['constraint_mode'] = 'mean_bin'
            df['constraint_metric'] = df['binned_count_accuracy_macro'].astype(float)
            df['metric_source'] = str(csv_path)
            return df[['ratio', 'constraint_mode', 'constraint_metric', 'metric_source']]
    return pd.DataFrame(columns=['ratio', 'constraint_mode', 'constraint_metric', 'metric_source'])


def build_mean_bin_metrics_for_config(dct_label, bin_size_samples, sample_rate, per_ratio_tables, baseline_mode, root=ROOT):
    if 'ratio1baseline' in str(baseline_mode):
        ratio1_df = load_ratio1baseline_week4_metrics_for_config(dct_label, bin_size_samples, sample_rate, root=root)
        if not ratio1_df.empty:
            return ratio1_df
        return synthesize_ratio1baseline_mean_bin_from_week5(per_ratio_tables)
    return load_legacy_week4_ratio_metrics_for_config(dct_label, bin_size_samples, sample_rate, root=root)


def build_per_ratio_tables(config_cache_df):
    per_ratio = {}
    for _, row in config_cache_df.iterrows():
        table_df = pd.DataFrame(row['payload']['per_neuron_table'])
        if table_df.empty:
            continue
        table_df = table_df.copy()
        table_df['neuron_id'] = table_df['neuron_id'].astype(int)
        table_df['ratio'] = float(row['ratio'])
        table_df['baseline_mode'] = row['baseline_mode']
        table_df['pipeline_version'] = row['pipeline_version']
        table_df['cache_path'] = str(row['cache_path'])
        per_ratio[float(row['ratio'])] = table_df
    return per_ratio


def all_neuron_constraint_metrics(per_ratio_tables, min_baseline_spikes=MIN_BASELINE_SPIKES):
    rows = []
    for ratio, table_df in sorted(per_ratio_tables.items()):
        valid = table_df[table_df['baseline_spikes'] >= min_baseline_spikes].copy()
        if valid.empty:
            metric = np.nan
            pass_fraction = np.nan
            n_valid = 0
        else:
            metric = float(valid['binned_count_accuracy'].min())
            pass_fraction = float((valid['binned_count_accuracy'] >= TARGET_ACCURACY).mean())
            n_valid = int(len(valid))
        rows.append({
            'ratio': float(ratio),
            'constraint_mode': 'all_neuron',
            'constraint_metric': metric,
            'pass_fraction': pass_fraction,
            'n_valid_neurons': n_valid,
            'metric_source': 'week5_per_neuron_cache',
        })
    return pd.DataFrame(rows)


def select_ratio_for_constraint(metric_df, target=TARGET_ACCURACY):
    metric_df = metric_df.dropna(subset=['constraint_metric']).sort_values('ratio')
    feasible = metric_df[metric_df['constraint_metric'] >= target]
    if not feasible.empty:
        return float(feasible['ratio'].iloc[0]), True
    if metric_df.empty:
        return np.nan, False
    best_idx = metric_df['constraint_metric'].idxmax()
    return float(metric_df.loc[best_idx, 'ratio']), False


def select_sensitive_robust(per_ratio_tables, selection_ratio):
    if not np.isfinite(selection_ratio):
        selection_ratio = sorted(per_ratio_tables.keys())[0]
    closest_ratio = min(per_ratio_tables.keys(), key=lambda r: abs(float(r) - float(selection_ratio)))
    selection_df = per_ratio_tables[closest_ratio]
    selection_df = selection_df[selection_df['baseline_spikes'] >= MIN_BASELINE_SPIKES].copy()
    sensitive = selection_df.sort_values(['binned_count_accuracy', 'baseline_spikes'], ascending=[True, False]).iloc[0]
    robust = selection_df.sort_values(['binned_count_accuracy', 'baseline_spikes'], ascending=[False, False]).iloc[0]
    return closest_ratio, int(sensitive['neuron_id']), int(robust['neuron_id'])


def build_selected_neuron_curves(per_ratio_tables, sensitive_id, robust_id):
    rows = []
    for ratio, table_df in sorted(per_ratio_tables.items()):
        selected = table_df[table_df['neuron_id'].isin([sensitive_id, robust_id])].copy()
        selected['neuron_type'] = np.where(selected['neuron_id'] == sensitive_id, 'most sensitive', 'most robust')
        rows.append(selected)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


## Plot Functions

In [ ]:
def plot_template_amplitude_projection(proj_df, selected_ids, title):
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    size = 30 + 220 * (proj_df['template_ptp_max'] / max(proj_df['template_ptp_max'].max(), 1e-12))
    sc = ax.scatter(
        proj_df['template_shape_pc1'],
        proj_df['template_shape_pc2'],
        c=proj_df['peak_depth_um'],
        s=size,
        cmap='viridis',
        alpha=0.82,
        edgecolors='none',
    )
    labels = {'most sensitive': 'crimson', 'most robust': 'seagreen'}
    for label, neuron_id in selected_ids.items():
        row = proj_df[proj_df['neuron_id'] == neuron_id]
        if row.empty:
            continue
        ax.scatter(row['template_shape_pc1'], row['template_shape_pc2'], s=210, facecolors='none', edgecolors=labels[label], linewidths=2.2, label=f'{label} {neuron_id}')
        ax.annotate(str(neuron_id), (float(row['template_shape_pc1'].iloc[0]), float(row['template_shape_pc2'].iloc[0])), textcoords='offset points', xytext=(6, 6), fontsize=9, color=labels[label])
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('dominant-channel peak depth (um)')
    ax.set_title(title)
    ax.set_xlabel('centered local template shape PC1')
    ax.set_ylabel('centered local template shape PC2')
    ax.grid(alpha=0.2)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

def plot_selected_neuron_curves(curve_df, selection_ratio, title):
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    styles = {
        'most sensitive': {'color': 'crimson', 'marker': 'o'},
        'most robust': {'color': 'seagreen', 'marker': 's'},
    }
    for neuron_type, group in curve_df.groupby('neuron_type', sort=False):
        group = group.sort_values('ratio')
        neuron_id = int(group['neuron_id'].iloc[0])
        ax.plot(group['ratio'], group['binned_count_accuracy'] * 100.0, linewidth=2.4, markersize=7, label=f'{neuron_type} neuron {neuron_id}', **styles[neuron_type])
    ax.axhline(TARGET_ACCURACY * 100.0, color='gray', linestyle=':', linewidth=1.1, label=f'target={TARGET_ACCURACY:.0%}')
    ax.axvline(selection_ratio, color='gray', linestyle='--', linewidth=1.0, alpha=0.75, label=f'selection ratio={selection_ratio:.2f}')
    ax.set_title(title)
    ax.set_xlabel('keep ratio')
    ax.set_ylabel('per-neuron binned-count accuracy (%)')
    ax.set_ylim(-2, 102)
    ax.set_xticks(sorted(curve_df['ratio'].unique()))
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()


def plot_overall_keep_ratio(metric_df, selected_ratio, feasible, title):
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    metric_df = metric_df.sort_values('ratio')
    ax.plot(metric_df['ratio'], metric_df['constraint_metric'] * 100.0, marker='o', linewidth=2.4, color='steelblue')
    ax.axhline(TARGET_ACCURACY * 100.0, color='crimson', linestyle='--', linewidth=1.2, label=f'target={TARGET_ACCURACY:.0%}')
    if np.isfinite(selected_ratio):
        ax.axvline(selected_ratio, color='darkorange' if feasible else 'gray', linestyle=':', linewidth=1.5, label=f"{'min feasible' if feasible else 'best observed'} ratio={selected_ratio:.2f}")
    ax.set_title(title)
    ax.set_xlabel('keep ratio')
    ax.set_ylabel('constraint metric accuracy (%)')
    ax.set_ylim(-2, 102)
    ax.set_xticks(sorted(metric_df['ratio'].unique()))
    ax.grid(alpha=0.25)
    ax.legend(loc='best')
    plt.tight_layout()
    plt.show()


## Alternative Local Template Projections

These panels compare several projection methods built from the same peak-channel-centered local multi-channel template window. Absolute probe depth is not part of the projected feature vector; it is only used as the color scale.


In [ ]:
PLOT_ALTERNATIVE_TEMPLATE_PROJECTIONS = True


def normalize_feature_rows(features):
    features = np.asarray(features, dtype=np.float32)
    features = features - features.mean(axis=1, keepdims=True)
    scale = np.linalg.norm(features, axis=1, keepdims=True)
    scale[scale <= 0] = 1.0
    return features / scale


def pca_2d_scores(features):
    features = normalize_feature_rows(features)
    centered = features - features.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    scores = centered @ vt[:2].T
    if np.nanmedian(scores[:, 0]) < 0:
        scores[:, 0] *= -1
    if np.nanmedian(scores[:, 1]) < 0:
        scores[:, 1] *= -1
    return scores.astype(np.float32)


def cosine_mds_2d_scores(features):
    features = normalize_feature_rows(features)
    cosine_similarity = np.clip(features @ features.T, -1.0, 1.0)
    distance = 1.0 - cosine_similarity
    distance2 = distance ** 2
    n = distance2.shape[0]
    center = np.eye(n, dtype=np.float32) - np.ones((n, n), dtype=np.float32) / n
    gram = -0.5 * center @ distance2 @ center
    eigvals, eigvecs = np.linalg.eigh(gram)
    order = np.argsort(eigvals)[::-1][:2]
    eigvals = np.maximum(eigvals[order], 0.0)
    scores = eigvecs[:, order] * np.sqrt(eigvals)[None, :]
    if np.nanmedian(scores[:, 0]) < 0:
        scores[:, 0] *= -1
    if np.nanmedian(scores[:, 1]) < 0:
        scores[:, 1] *= -1
    return scores.astype(np.float32)


def build_alternative_template_projection_frames(templates, ops, base_df, radius=LOCAL_CHANNEL_RADIUS):
    local_templates, *_ = extract_centered_local_templates(templates, radius=radius)
    n_templates = local_templates.shape[0]

    local_template_features = local_templates.reshape(n_templates, -1)
    temporal_derivative_features = np.diff(local_templates, axis=1).reshape(n_templates, -1)
    footprint_features = np.ptp(local_templates, axis=1)
    center_waveform_features = local_templates[:, :, radius]

    projection_specs = [
        ('Local template PCA', pca_2d_scores(local_template_features), 'local spatiotemporal shape'),
        ('Temporal-derivative PCA', pca_2d_scores(temporal_derivative_features), 'waveform dynamics'),
        ('Spatial-footprint PCA', pca_2d_scores(footprint_features), 'relative multi-channel footprint'),
        ('Cosine MDS', cosine_mds_2d_scores(local_template_features), 'cosine distance on local template shape'),
    ]

    frames = []
    for method, scores, feature_note in projection_specs:
        frame = base_df[['neuron_id', 'peak_depth_um', 'template_ptp_max', 'template_energy']].copy()
        frame['projection_method'] = method
        frame['feature_note'] = feature_note
        frame['alt_projection_x'] = scores[:, 0]
        frame['alt_projection_y'] = scores[:, 1]
        frames.append(frame)
    return frames


alternative_template_projection_frames = build_alternative_template_projection_frames(
    templates_base,
    ops_base,
    template_projection_df,
)


def plot_template_projection_alternatives(alternative_frames, selected_ids, title):
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 10.0), sharex=False, sharey=False)
    axes = axes.ravel()
    color_min = min(float(frame['peak_depth_um'].min()) for frame in alternative_frames)
    color_max = max(float(frame['peak_depth_um'].max()) for frame in alternative_frames)
    labels = {'most sensitive': 'crimson', 'most robust': 'seagreen'}
    last_scatter = None

    for ax, frame in zip(axes, alternative_frames):
        size = 28 + 210 * (frame['template_ptp_max'] / max(frame['template_ptp_max'].max(), 1e-12))
        last_scatter = ax.scatter(
            frame['alt_projection_x'],
            frame['alt_projection_y'],
            c=frame['peak_depth_um'],
            s=size,
            cmap='viridis',
            vmin=color_min,
            vmax=color_max,
            alpha=0.82,
            edgecolors='none',
        )
        for label, neuron_id in selected_ids.items():
            row = frame[frame['neuron_id'] == neuron_id]
            if row.empty:
                continue
            ax.scatter(row['alt_projection_x'], row['alt_projection_y'], s=180, facecolors='none', edgecolors=labels[label], linewidths=2.0, label=f'{label} {neuron_id}')
            ax.annotate(str(neuron_id), (float(row['alt_projection_x'].iloc[0]), float(row['alt_projection_y'].iloc[0])), textcoords='offset points', xytext=(5, 5), fontsize=8.5, color=labels[label])
        method = str(frame['projection_method'].iloc[0])
        note = str(frame['feature_note'].iloc[0])
        ax.set_title(f'{method}\n{note}', fontsize=10)
        ax.set_xlabel('projection 1')
        ax.set_ylabel('projection 2')
        ax.grid(alpha=0.2)
        ax.legend(loc='best', fontsize=8)

    fig.suptitle(title, y=0.995)
    if last_scatter is not None:
        cbar = fig.colorbar(last_scatter, ax=axes.tolist(), fraction=0.025, pad=0.02)
        cbar.set_label('dominant-channel peak depth (um)')
    plt.show()


## Render Figures

For each discovered configuration and each constraint mode, the notebook renders:

1. improved maximum-amplitude probe projection;
2. most sensitive / robust neuron accuracy curves across keep ratios;
3. overall keep-ratio constraint curve.

If a Week 4 mean-bin aggregate is not available for a configuration, the mean-bin constraint is reported as unavailable for that configuration rather than estimated from incompatible data.

In [ ]:
render_summary_rows = []
all_curve_rows = []

for config_key, raw_config_df in week5_cache_index_df.groupby(['dct_window_label', 'bin_size_samples', 'sample_rate'], sort=True):
    dct_label, bin_size_samples, sample_rate = config_key
    config_df = select_cache_family_for_config(raw_config_df).sort_values('ratio').reset_index(drop=True)
    baseline_mode = '+'.join(sorted(config_df['baseline_mode'].unique()))
    per_ratio_tables = build_per_ratio_tables(config_df)
    if not per_ratio_tables:
        continue

    display(Markdown(f'## Config: DCT={dct_label}, bin={int(bin_size_samples) / float(sample_rate):g}s, baseline={baseline_mode}'))

    constraint_metric_frames = []
    mean_bin_df = build_mean_bin_metrics_for_config(dct_label, int(bin_size_samples), float(sample_rate), per_ratio_tables, baseline_mode)
    if not mean_bin_df.empty:
        constraint_metric_frames.append(mean_bin_df)
    else:
        print('mean_bin constraint unavailable: no matching Week 4 per-bin-mean aggregate table found for this config.')

    all_neuron_df = all_neuron_constraint_metrics(per_ratio_tables)
    constraint_metric_frames.append(all_neuron_df)

    for metric_df in constraint_metric_frames:
        mode = metric_df['constraint_mode'].iloc[0]
        display(Markdown(f'### Constraint: {mode}'))
        display(metric_df)

        selected_ratio, feasible = select_ratio_for_constraint(metric_df)
        selection_ratio, sensitive_id, robust_id = select_sensitive_robust(per_ratio_tables, selected_ratio if np.isfinite(selected_ratio) else PREFERRED_SELECTION_RATIO)
        selected_ids = {'most sensitive': sensitive_id, 'most robust': robust_id}
        curve_df = build_selected_neuron_curves(per_ratio_tables, sensitive_id, robust_id)
        curve_df['constraint_mode'] = mode
        curve_df['dct_window_label'] = dct_label
        curve_df['bin_size_samples'] = int(bin_size_samples)
        curve_df['bin_size_seconds'] = int(bin_size_samples) / float(sample_rate)
        curve_df['baseline_mode'] = baseline_mode
        all_curve_rows.append(curve_df)

        title_prefix = f'{mode} | DCT={dct_label} | bin={int(bin_size_samples) / float(sample_rate):g}s | {baseline_mode}'
        plot_template_amplitude_projection(
            template_projection_df,
            selected_ids,
            title=f'Spike Template Shape PCA | {title_prefix}',
        )
        if PLOT_ALTERNATIVE_TEMPLATE_PROJECTIONS:
            plot_template_projection_alternatives(
                alternative_template_projection_frames,
                selected_ids,
                title=f'Alternative Local Template Projections | {title_prefix}',
            )
        plot_selected_neuron_curves(
            curve_df,
            selection_ratio,
            title=f'Most Sensitive / Robust Neurons | {title_prefix}',
        )
        plot_overall_keep_ratio(
            metric_df,
            selected_ratio,
            feasible,
            title=f'Overall Keep-Ratio Constraint | {title_prefix}',
        )

        curve_csv = ROOT / f'week7_selected_neuron_curves_{safe_config_tag(dct_label)}_{int(bin_size_samples)}sampbin_{baseline_mode}_{mode}.csv'
        curve_df.to_csv(curve_csv, index=False)
        render_summary_rows.append({
            'dct_window_label': dct_label,
            'bin_size_samples': int(bin_size_samples),
            'bin_size_seconds': int(bin_size_samples) / float(sample_rate),
            'baseline_mode': baseline_mode,
            'constraint_mode': mode,
            'target_accuracy': TARGET_ACCURACY,
            'selected_ratio': selected_ratio,
            'feasible': feasible,
            'selection_ratio_for_neurons': selection_ratio,
            'most_sensitive_neuron_id': sensitive_id,
            'most_robust_neuron_id': robust_id,
            'curve_csv': str(curve_csv),
        })

week7_render_summary_df = pd.DataFrame(render_summary_rows)
display(Markdown('## Render Summary'))
display(week7_render_summary_df)

if all_curve_rows:
    week7_all_selected_neuron_curves_df = pd.concat(all_curve_rows, ignore_index=True)
else:
    week7_all_selected_neuron_curves_df = pd.DataFrame()

summary_csv = ROOT / 'week7_constraint_render_summary.csv'
week7_render_summary_df.to_csv(summary_csv, index=False)
print('saved:', summary_csv)
